In [1]:
# 接入网盘数据

In [14]:
import pypff
from pathlib import Path

pst_path = Path("../data/raw/Van email/Van Inbox backup 05.14.21.pst")

pst = pypff.file()
pst.open(str(pst_path))

root = pst.get_root_folder()
print("Root folder:", root.name) # 文件成功打开会显示名字 显示None也代表打开了

Root folder: None


In [15]:
# 解析pst文件 确认该文件夹下面都包含什么文件
# 包括Inbox Sent Items 可能还有Archive / Deleted / Junk等
def show_folders(folder, level=0):
    folder_name = folder.name if folder.name else "[No Name]"
    print("  " * level + f"- {folder_name}")

    for i in range(folder.number_of_sub_folders):
        sub_folder = folder.get_sub_folder(i)
        show_folders(sub_folder, level + 1)

show_folders(root)

- [No Name]
  - SPAM Search Folder 2
  - Top of Outlook data file
    - Deleted Items
    - Inbox
  - Search Root


In [17]:
# 看每个folder当中有多少邮件
def show_folders_with_counts(folder, level=0):
    folder_name = folder.name if folder.name else "[No Name]"
    print("  " * level + f"- {folder_name} | messages: {folder.number_of_sub_messages}")

    for i in range(folder.number_of_sub_folders):
        sub_folder = folder.get_sub_folder(i)
        show_folders_with_counts(sub_folder, level + 1)

show_folders_with_counts(root)

- [No Name] | messages: 0
  - SPAM Search Folder 2 | messages: 0
  - Top of Outlook data file | messages: 0
    - Deleted Items | messages: 0
    - Inbox | messages: 11111
  - Search Root | messages: 0


In [29]:
# 根据文件夹名字找到固定的文件夹（这里其实就是要找inbox）
def find_folder_by_name(folder, target_name):
    if folder.name == target_name:
        return folder
    
    for i in range(folder.number_of_sub_folders):
        sub_folder = folder.get_sub_folder(i)
        result = find_folder_by_name(sub_folder, target_name)
        if result is not None:
            return result
    
    return None

inbox = find_folder_by_name(root, "Inbox")

print(inbox)
print("Inbox message count:", inbox.number_of_sub_messages if inbox else "Not found")

Inbox message count: 11111


# Email Dataset Structure (PST Parsing Output)

This document describes the structure of the parsed email dataset extracted from PST files.  
Each row represents **one email message**.

---

## Table Schema

| Column Name    | Data Type | Description |
|----------------|----------|-------------|
| `id`           | Integer  | Unique identifier for each email (generated sequentially during parsing) |
| `subject`      | String   | Email subject line |
| `sender`       | String   | Display name of the sender |
| `sender_email` | String   | Sender's email address |
| `to`           | String   | Recipient(s) of the email (semicolon-separated if multiple) |
| `date`         | Datetime | Email delivery timestamp |
| `body`         | String   | Raw email body text (original, unprocessed) |
| `body_clean`   | String   | Cleaned email body text (processed for analysis) |

---

In [31]:
import pandas as pd
import re
from typing import List, Dict, Any

In [32]:
def safe_decode(value):
    """
    Safely decode PST fields into readable Python strings.
    Handles None, bytes, and normal string-like values.
    """
    if value is None:
        return ""

    if isinstance(value, bytes):
        for enc in ("utf-8", "latin-1", "cp1252"):
            try:
                return value.decode(enc, errors="ignore")
            except Exception:
                continue
        return ""

    return str(value)

In [34]:
def clean_text(text: str) -> str:
    """
    Basic text cleaning for email body.
    Keeps it simple for now:
    - remove line breaks
    - remove URLs
    - collapse multiple spaces
    """
    if not text:
        return ""

    text = safe_decode(text)
    text = re.sub(r"\r\n|\r|\n", " ", text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [35]:
def extract_body(message) -> str:
    """
    Extract email body with simple fallback logic:
    1. plain_text_body
    2. html_body
    """
    body = ""

    try:
        body = message.plain_text_body
    except Exception:
        body = None

    if not body:
        try:
            body = message.html_body
        except Exception:
            body = None

    return safe_decode(body)

In [40]:
def extract_email_from_headers(headers, key):
    if not headers:
        return ""

    headers = safe_decode(headers)

    # From: xxx <email>
    match = re.search(fr"{key}:.*?<(.*?)>", headers, re.IGNORECASE)
    if match:
        return match.group(1)

    # From: email
    match = re.search(fr"{key}:\s*(\S+@\S+)", headers, re.IGNORECASE)
    if match:
        return match.group(1)

    return ""

In [41]:
def extract_recipients(message):
    recipients = []

    try:
        for i in range(message.number_of_recipients):
            r = message.get_recipient(i)
            email = safe_decode(r.email_address)
            if email:
                recipients.append(email)
    except:
        pass

    return "; ".join(recipients)

In [42]:
def parse_inbox_to_df(inbox, max_messages=None):
    records = []

    total = inbox.number_of_sub_messages
    limit = min(total, max_messages) if max_messages else total

    for i in range(limit):
        try:
            message = inbox.get_sub_message(i)

            # headers（关键）
            headers = getattr(message, "transport_headers", "")

            # sender email（多重 fallback）
            sender_email = (
                safe_decode(getattr(message, "sender_email_address", ""))
                or extract_email_from_headers(headers, "From")
            )

            # to（优先 recipients，其次 headers）
            to = (
                extract_recipients(message)
                or extract_email_from_headers(headers, "To")
            )

            # body
            body = extract_body(message)
            body_clean = clean_text(body)

            record = {
                "id": i,
                "subject": safe_decode(getattr(message, "subject", "")),
                "sender": safe_decode(getattr(message, "sender_name", "")),
                "sender_email": sender_email,
                "to": to,
                "date": getattr(message, "delivery_time", None),
                "body": body,
                "body_clean": body_clean,
            }

            records.append(record)

        except Exception as e:
            print(f"⚠️ Skipping message {i}: {e}")

    df = pd.DataFrame(records, columns=[
        "id",
        "subject",
        "sender",
        "sender_email",
        "to",
        "date",
        "body",
        "body_clean",
    ])

    return df

In [48]:
df = parse_inbox_to_df(inbox)
df.head()

,id,subject,sender,sender_email,to,date,body,body_clean
0,0,Canceled: Weekly Meeting to Discuss Business &...,Judy Tse,judy.tse@promab.com,john@promab.com,2021-04-22 19:02:43.876892,\r\n,
1,1,"JUDY OUT OF OFFICE FOR VACATION MAY 19-21, 202...",Judy Tse,judy.tse@promab.com,john@promab.com,2021-04-19 23:12:26.137195,Hi All.\r\n\r\nI will be taking some time off ...,Hi All. I will be taking some time off (3 days...
2,2,"Tim Off Wednesday, 04/14/2021",Timothy Nguyen,timothy.nguyen@promab.com,,2021-04-13 22:02:22.134260,"Hi all,\r\n\r\n \r\n\r\nFriendly reminder that...","Hi all, Friendly reminder that I will be takin..."
3,3,Canceled: Weekly Meeting to Discuss Business &...,Judy Tse,judy.tse@promab.com,john@promab.com,2021-03-23 16:36:21.708864,\r\n,
4,4,Weekly Meeting Move to Wed. March 24th 3:00 PM,Timothy Nguyen,timothy.nguyen@promab.com,william.chen@promab.com,2021-03-23 16:34:28.291447,"Hi all, just sending a meeting invitation for ...","Hi all, just sending a meeting invitation for ..."


In [49]:
# 测试
df.sample(5)[["subject", "sender_email", "to", "body_clean"]]

,subject,sender_email,to,body_clean
3198,RE: CD147,john@promab.com,van.dang@promab.com,"Hi Van, Feb is OK。 You told me you want to con..."
5466,,van.dang@promab.com,van.dang@promab.com,
8315,Re: viruses plasmid,timothy.nguyen@promab.com,van.dang@promab.com,Sure. The operator didn’t really sound like th...
5278,RE: Quote for CAR-T cells,nd@cellvie.bio,van.dang@promab.com,"Dear Van, Great, thank you very much. Best reg..."
1132,RE: [EXTERNAL]RE: PO 23253,timothy.nguyen@promab.com,lruff@regulusrx.com,"Hi Laura, I will try to arrange for shipment t..."


In [50]:
print("Total emails:", len(df))
print("\nMissing values:")
print(df.isna().sum())

print("\nEmpty body:")
print((df["body_clean"] == "").sum())

Total emails: 11111

Missing values:
id              0
subject         0
sender          0
sender_email    0
to              0
date            0
body            0
body_clean      0
dtype: int64

Empty body:
650


In [51]:
df["body_len"] = df["body_clean"].apply(len)
df["body_len"].describe()

count     11111.000000
mean       5444.362074
std        9267.042708
min           0.000000
25%         763.000000
50%        2160.000000
75%        6148.000000
max      108788.000000
Name: body_len, dtype: float64